# Conservative 2D regrid — basics

`.conservative` uses fast axis-factored 1D overlap and only works on 1D-separable
rectilinear grids. `ConservativeRegridder` / `.regrid.conservative_2d(...)` fills in the
non-1D-separable cases — curvilinear 2D coordinates, unstructured meshes, arbitrary
polygon targets — by computing the full 2D cell-polygon intersection via shapely. It
also exposes a spherical-area mode for correct weights on global lat/lon grids.

This notebook shows the basic workflow on a rectilinear grid (to keep the first example
simple and compare directly against the factored path). Curvilinear and unstructured use
cases are in the sibling demos.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import xarray_regrid  # registers the `.regrid` accessor
from xarray_regrid import ConservativeRegridder

np.set_printoptions(precision=5, suppress=True)

## Synthetic source field

We use $f(\lambda, \phi) = \cos^2(\phi)$ on a 1° global grid. Its true integral over the unit
sphere is $8\pi/3$, which gives us a check on the regridder.

In [ ]:
lat = np.linspace(-89.5, 89.5, 180)
lon = np.linspace(-179.5, 179.5, 360)
field = (np.cos(np.deg2rad(lat))**2)[:, None] * np.ones(lon.size)[None, :]
src = xr.DataArray(
    field,
    dims=("latitude", "longitude"),
    coords={"latitude": lat, "longitude": lon},
    name="cos2_lat",
)
src

In [ ]:
src.plot(figsize=(8, 3.5), cmap="viridis")
plt.title("source: cos²(lat) on a 1° grid")
plt.tight_layout()

## Target grid

A coarser 3° grid. We'll build an empty `xr.Dataset` with just the target coords;
`xarray-regrid` identifies the grid from `latitude` and `longitude`.

In [ ]:
target = xr.Dataset(coords={
    "latitude": np.linspace(-88.5, 88.5, 60),
    "longitude": np.linspace(-178.5, 178.5, 120),
})

## Regrid via the `.regrid.conservative_2d` accessor

`spherical=True` applies an analytic Lambert cylindrical equal-area projection
to the cell edges before intersecting — correct spherical area weights at the
same cost as the planar fast path. Leave it off when your coords aren't geographic degrees.

In [ ]:
regridded = src.regrid.conservative_2d(
    target, x_coord="longitude", y_coord="latitude", spherical=True
)
regridded

In [ ]:
regridded.plot(figsize=(8, 3.5), cmap="viridis")
plt.title("regridded: 3° target")
plt.tight_layout()

## Mass-conservation diagnostic

A correct conservative regrid preserves the area-weighted integral (on the sphere).
We compute true spherical cell areas for each grid and compare the integrals.

In [ ]:
def sph_integral(da):
    lat_r = np.deg2rad(da.latitude.values)
    dlat = np.gradient(lat_r)
    dlon = np.gradient(np.deg2rad(da.longitude.values))
    area = (np.sin(lat_r + dlat/2) - np.sin(lat_r - dlat/2))[:, None] * dlon[None, :]
    return float((da.values * area).sum())

true_val = 8 * np.pi / 3
src_sum = sph_integral(src)
out_sum = sph_integral(regridded)
print(f"true sphere integral : {true_val:.6f}")
print(f"source integral      : {src_sum:.6f}  (err {src_sum - true_val:+.2e})")
print(f"regridded integral   : {out_sum:.6f}  (err {out_sum - true_val:+.2e})")

## Spherical vs planar vs the axis-factored path

`spherical=False` uses raw lat/lon planar geometry — poor near the poles.
The existing `.conservative` method applies an analytic sin-weighting to the factored
1D overlap. On lat/lon grids, `spherical=True` reproduces that accuracy while also
working on curvilinear and unstructured targets the factored path can't express.

In [ ]:
def err(fn):
    return abs(sph_integral(fn()) - true_val)

results = {
    "2d, spherical=True ": err(lambda: src.regrid.conservative_2d(
        target, x_coord="longitude", y_coord="latitude", spherical=True)),
    "2d, spherical=False": err(lambda: src.regrid.conservative_2d(
        target, x_coord="longitude", y_coord="latitude", spherical=False)),
    "factored (.conservative)": err(lambda: src.regrid.conservative(
        target, latitude_coord="latitude")),
}
for k, v in results.items():
    print(f"{k}:  |error| = {v:.2e}")